In [0]:
%run ./04_Function_Book

In [0]:
# Define dq_config: specifies all quality rules for orders (nulls, formats, date ranges, PK)
# Used as input to validation functions

dq_config = {
    "table_name": "orders",

    # NULL CHECKS
    "null_checks": {
        "order_id": 0,
        "customer_id": 0,
        "order_status": 0,
        "order_purchase_timestamp": 0,
        "order_approved_at": 0,
        "order_delivered_carrier_date": 0,
        "order_delivered_customer_date": 0,
        "order_estimated_delivery_date": 0
    },

    # FORMAT CHECKS
    "format_checks": {
        # REGEX, DATATYPE, and TIMESTAMP checks with zero tolerance for violations
        "regex": {
            "order_id": {
                "pattern": r"^[A-Za-z0-9]{12,}$",
                "threshold": 0
            },
            "customer_id": {
                "pattern": r"^[A-Za-z0-9]{12,}$",
                "threshold": 0
            }
        },
        "datatype_check": {
            # Specifies expected type for each column
            "order_id": {"expected": "string", "threshold": 0},
            "customer_id": {"expected": "string", "threshold": 0},
            "order_status": {"expected": "string", "threshold": 0},
            "order_purchase_timestamp": {"expected": "timestamp", "threshold": 0},
            "order_approved_at": {"expected": "timestamp", "threshold": 0},
            "order_delivered_carrier_date": {"expected": "timestamp", "threshold": 0},
            "order_delivered_customer_date": {"expected": "timestamp", "threshold": 0},
            "order_estimated_delivery_date": {"expected": "timestamp", "threshold": 0}
        },
        "timestamp_check": {
            # Specifies expected timestamp format for each column
            "order_purchase_timestamp": {"format": "yyyy-MM-dd HH:mm:ss", "threshold": 0},
            "order_approved_at": {"format": "yyyy-MM-dd HH:mm:ss", "threshold": 0},
            "order_delivered_carrier_date": {"format": "yyyy-MM-dd HH:mm:ss", "threshold": 0},
            "order_delivered_customer_date": {"format": "yyyy-MM-dd HH:mm:ss", "threshold": 0},
            "order_estimated_delivery_date": {"format": "yyyy-MM-dd", "threshold": 0}
        }
    },

    # DATE RANGE CHECKS
    "date_range_checks": {
        "order_purchase_timestamp": {
            "min": "2016-09-04",
            "max": "2018-10-18",
            "threshold": 0
        }
    },

    # PRIMARY KEY CHECK
    "primary_key": {
        "column": "order_id",
        "threshold": 0
    }
}


In [0]:
# Define a function to apply all DQ checks, aggregate as metrics DataFrame
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

def generate_metrics_df(df, config):
    # Accumulate results from null, format, date, pk checks
    all_results = []
    all_results += run_null_checks(df, config)
    all_results += run_format_checks(df, config)
    all_results += run_date_range_checks(df, config)
    all_results += run_pk_check(df, config)
    # Tag each row with the table name
    for result in all_results:
        result["table_name"] = config["table_name"]
    # Build metrics DataFrame and timestamp
    metrics_df = spark.createDataFrame(Row(**r) for r in all_results) \
        .withColumn("run_time", current_timestamp())
    return metrics_df

In [0]:
# Load the orders silver table and run configured DQ routines
# Produces a metrics DataFrame for summary and reporting
df_orders = spark.table("retail_catalog.silver.orders")
metrics_df = generate_metrics_df(df_orders, dq_config)

In [0]:
# Return the metrics DataFrame to the master/orchestrator notebook for consolidation
metrics_df

DataFrame[column_name: string, check_name: string, metric_type: string, metric_value: bigint, threshold: bigint, status: string, table_name: string, run_time: timestamp]